# 章节练习

这一章覆盖了 SDPA 的接口与 dispatch、FlashAttention 的 IO 融合、GQA 的 KV head 缩减、稀疏注意力的结构化跳过、SFT document mask 的语义、TorchTitan dataloader 的文档边界、Configurable 机制、torch_npu 的 dispatcher，以及固定 `S=4096` 的性能计时和 trace。

做练习时：**写清楚你用的是哪一层证据。** 说"用了 FlashAttention"不够——你看到的是配置里的 `FLASH_ATTENTION` 枚举，还是 trace 里的 kernel 名称，还是计时数据里的加速比？这些证据的可信度不一样，练习的目的之一就是训练你区分它们。

## 综合练习

1. **Score 矩阵计算**：本章固定 B=2、S=4096、Nq=16、D=128、BF16。若 eager 路径物化 QK^T，逻辑 score 矩阵占多少 MiB？如果改用 GQA 且 Nkv=8，score 矩阵的大小会变吗？为什么？

2. **配置追踪**：下面是一段 TorchTitan 的 attention 配置代码。inner_attention 最终 build 出来的是什么模块？如果换成 FlexAttention，需要改哪个对象？换成 NPUVarlenAttention 之后，trainer 的行为会有什么连带变化？

3. **性能证据判读**：在固定 S=4096 的 trace 中，SDPA 与 FusionAttention V3 的对应 device time 基本一致，但 CPU 侧调用链不同。这个结果最多能支持什么结论？哪些结论仍需要独立的 step wall-time 才能支持？

4. **实验设计审查**：有人改了 05.03 的 benchmark 代码，把 warmup 从 5 次改成了 0 次，把 `torch.npu.synchronize()` 从计时区间前后都删掉了，并且只跑了一遍 S=4096。他说"FusionAttention V3 比 SDPA 快 3.2 倍"。这个结论能信吗？逐条指出实验设计的问题。

5. **Trace 判读**：你拿到一份 profiler trace，eager 路径的 kernel 列表有 40+ 个条目，SDPA 路径只有 4 个。但 SDPA 的四个 kernel 里有一个叫 `aten::scaled_dot_product_attention`。这是否说明 SDPA 没有用上融合 kernel？如果 trace 里出现 `npu_fusion_attention_v3`，你的结论又是什么？

6. **写一份性能结论**：根据 05.03 的实验结果（或你实际跑出来的数据），写一段不超过 200 字的性能结论。要求：写明固定 shape `(B=2,S=4096,Nq=16,Nkv=8,D=128)`、dtype、硬件、软件版本、比较对象和测量口径（forward 还是 fwd+bwd、是否包含 warmup、重复次数）、主要发现，以及一条明确的适用范围限制。

## 答完题之后

回顾你的答案，标出哪些结论只在固定长度 causal attention 的场景下成立。到了第 6 章的变长注意力（不同文档有不同长度、需要 document boundary mask），这些结论里哪些需要重新验证？为什么？

## 参考答案

先独立完成练习，再对照 [解释型参考答案](answer/05.06_answer.txt)。答案给出推理和证据边界，不只列结论。

In [ ]:
!cat ./answer/05.06_answer.txt
